# Ruby AI Pull Request Reviewer

This notebook runs the complete pipeline:

GitHub PR → diff parsing → repository checkout → RuboCop → RAG/FAISS → OpenAI → structured review → merged findings.


## 0. Configuration

Before running this notebook, make sure your project contains:

```text
review-ai/
├── notebook.ipynb
├── requirements.txt
├── .env
├── knowledge/
│   ├── rails.md
│   ├── security.md
│   ├── performance.md
│   ├── clean_code.md
│   └── testing.md
└── src/
    ├── __init__.py
    ├── config.py
    ├── github.py
    ├── parser.py
    ├── rag.py
    ├── rubocop.py
    ├── openai_client.py
    ├── reviewer.py
    ├── schema.py
    └── aggregator.py
```

Your `.env` should contain:

```text
OPENAI_API_KEY=your_key_here
```

In [1]:
from pathlib import Path
import sys
import os

# Make the project root importable when this notebook is run from the project directory.
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

Project root: /Users/renzodiaz/workspace/backend/python/review-ai
Python: 3.14.6


## 1. Imports

Run this after installing `requirements.txt`.

In [2]:
from src.github import (
    download_pr,
    clone_pr_repository,
    get_pr_metadata,
    parse_pr_url,
)

from src.parser import parse_diff
from src.rag import load_vector_db, retrieve
from src.rubocop import run_rubocop, format_offenses
from src.reviewer import review
from src.aggregator import rubocop_to_issues, merge_review

print("Imports OK")

/Users/renzodiaz/.local/share/mise/installs/python/3.14.6/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 2. Choose the Pull Request

For the first demo, use a public GitHub repository. Private repositories require GitHub authentication.

In [3]:
PR_URL = "https://github.com/renzodiaz/notes-api/pull/4"

print("PR:", PR_URL)

PR: https://github.com/renzodiaz/notes-api/pull/4


## 3. Validate the PR URL

In [4]:
pr_info = parse_pr_url(PR_URL)

print("Owner:", pr_info["owner"])
print("Repository:", pr_info["repo"])
print("PR number:", pr_info["number"])

Owner: renzodiaz
Repository: notes-api
PR number: 4


## 4. Get PR metadata from GitHub

In [5]:
metadata = get_pr_metadata(PR_URL)

for key, value in metadata.items():
    print(f"{key}: {value}")

owner: renzodiaz
repo: notes-api
number: 4
clone_url: https://github.com/renzodiaz/notes-api.git
head_branch: feat/base-authentication
head_sha: 040e8baebccde79ea0f682f02dfc36f77763d5c4
base_branch: main


## 5. Download the PR diff

The diff is the primary input for understanding what changed.

In [6]:
diff = download_pr(PR_URL)

print(f"Downloaded {len(diff):,} characters.")
print(f"Diff lines: {len(diff.splitlines()):,}")

Downloaded 5,625 characters.
Diff lines: 176


## 6. Inspect the raw diff

This is optional, but useful while developing.

In [7]:
print(diff[:8000])

diff --git a/Gemfile b/Gemfile
index 00aa1f8..a09c965 100644
--- a/Gemfile
+++ b/Gemfile
@@ -10,7 +10,7 @@ gem "puma", ">= 5.0"
 # gem "jbuilder"
 
 # Use Active Model has_secure_password [https://guides.rubyonrails.org/active_model_basics.html#securepassword]
-# gem "bcrypt", "~> 3.1.7"
+gem "bcrypt", "~> 3.1.7"
 
 # Windows does not include zoneinfo files, so bundle the tzinfo-data gem
 gem "tzinfo-data", platforms: %i[ windows jruby ]
diff --git a/Gemfile.lock b/Gemfile.lock
index bfbf0f0..55df6f9 100644
--- a/Gemfile.lock
+++ b/Gemfile.lock
@@ -77,6 +77,7 @@ GEM
       uri (>= 0.13.1)
     ast (2.4.3)
     base64 (0.3.0)
+    bcrypt (3.1.22)
     bcrypt_pbkdf (1.1.2)
     bigdecimal (4.1.2)
     bootsnap (1.25.0)
@@ -345,6 +346,7 @@ PLATFORMS
   x86_64-linux-musl
 
 DEPENDENCIES
+  bcrypt (~> 3.1.7)
   bootsnap
   brakeman
   bundler-audit
@@ -376,6 +378,7 @@ CHECKSUMS
   activesupport (8.1.3.1) sha256=85458765f25ea48b9019c46b6bb3fa5683197bf4280d9f06710a6e8d7a831376
   ast (2.4.3) 

## 7. Parse the diff

The parser identifies changed files, Ruby files, production files, tests, and diff hunks.

In [8]:
parsed = parse_diff(diff)

print("Changed files:", len(parsed["changed_files"]))
print("Ruby files:", len(parsed["ruby_files"]))
print("Production Ruby files:", len(parsed["production_files"]))
print("Test files:", len(parsed["test_files"]))

Changed files: 10
Ruby files: 7
Production Ruby files: 6
Test files: 1


## 8. Show changed files

In [9]:
for file in parsed["changed_files"]:
    print(
        f"{file['filename']} | "
        f"Ruby={file['is_ruby']} | "
        f"Test={file['is_test']} | "
        f"Hunks={len(file['hunks'])}"
    )

Gemfile | Ruby=False | Test=False | Hunks=1
Gemfile.lock | Ruby=False | Test=False | Hunks=3
app/controllers/api/v1/auth_controller.rb | Ruby=True | Test=False | Hunks=1
app/controllers/api/v1/secure_controller.rb | Ruby=True | Test=False | Hunks=1
app/models/user.rb | Ruby=True | Test=False | Hunks=1
config/routes.rb | Ruby=True | Test=False | Hunks=1
db/migrate/20260808154603_create_users.rb | Ruby=True | Test=False | Hunks=1
db/schema.rb | Ruby=True | Test=False | Hunks=1
test/fixtures/users.yml | Ruby=False | Test=True | Hunks=1
test/models/user_test.rb | Ruby=True | Test=True | Hunks=1


## 9. Show the Ruby code being reviewed

The reviewer uses added Ruby code plus removed code and surrounding hunk context.

In [10]:
changed_code = parsed["added_ruby_code"]

print(changed_code)


FILE: app/controllers/api/v1/auth_controller.rb

HUNK:
@@ -0,0 +1,13 @@

ADDED CODE:
module Api::V1
    class AuthController < SecureController
        def login
            user = User.find_by(email: params[:email])

            if user && user.authenticate(params[:password])
                render json: { user: user }, status: :ok
            end

            render json: { error: "Invalid email or password" }, status: :unauthorized
        end
    end
end

REMOVED CODE:


CONTEXT:
\ No newline at end of file


FILE: app/controllers/api/v1/secure_controller.rb

HUNK:
@@ -0,0 +1,4 @@

ADDED CODE:
module Api::V1
    class SecureController < ApplicationController
    end
end

REMOVED CODE:


CONTEXT:
\ No newline at end of file


FILE: app/models/user.rb

HUNK:
@@ -0,0 +1,3 @@

ADDED CODE:
class User < ApplicationRecord
    has_secure_password
end

REMOVED CODE:


CONTEXT:



FILE: config/routes.rb

HUNK:
@@ -5,6 +5,12 @@

ADDED CODE:
  namespace :api do
    namespace :v1 do
      post

## 10. Build the RAG vector database

Markdown engineering guidelines are chunked, embedded with Hugging Face, and stored in FAISS.

In [11]:
db = load_vector_db()

print("RAG vector database ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9628.97it/s]


RAG vector database ready.


## 11. Test RAG retrieval

This cell is shows we are using retrieving engineering knowledge.

In [12]:
query = changed_code
retrieved_documents = retrieve(db, query)

print(f"Retrieved {len(retrieved_documents)} knowledge chunks.\n")

for index, document in enumerate(retrieved_documents, start=1):
    print("=" * 70)
    print(f"RESULT {index}")
    print("SOURCE:", document.metadata.get("source"))
    print(document.page_content[:1500])
            

Retrieved 5 knowledge chunks.

RESULT 1
SOURCE: security.md
Good:

User.where(email: params[:email])

## Strong Parameters

Never use permit! when handling user input unless there is a very
specific and justified reason.

Prefer explicitly permitting only the parameters the application needs.

Good:

params.require(:user).permit(:name, :email)

## Authorization

Authentication and authorization are different concerns.

A user being authenticated does not mean they are authorized to
perform every action.
RESULT 2
SOURCE: security.md
# Ruby on Rails Security Guidelines

## Password Authentication

Never compare a plaintext password directly with a stored password.

Bad:

user.password == params[:password]

Prefer Rails authentication mechanisms such as has_secure_password
and authenticate.

## SQL Injection

Never interpolate user-controlled values directly into SQL strings.

Bad:

User.where("email = '#{params[:email]}'")

Prefer parameterized Active Record queries.

Good:

User.where(e

## 12. Clone the PR repository

RuboCop needs the actual source tree, not just the diff.

In [13]:
repo_path = clone_pr_repository(PR_URL)

print("Repository cloned to:")
print(repo_path)

Repository cloned to:
/var/folders/b1/d_q83c6s5t98lzfhlg1w5vhw0000gn/T/review_ai_21b909a_


## 13. Determine which Ruby files RuboCop should inspect

In [14]:
ruby_files = [
    file["filename"]
    for file in parsed["ruby_files"]
]

print("Ruby files:")
for filename in ruby_files:
    print("-", filename)

Ruby files:
- app/controllers/api/v1/auth_controller.rb
- app/controllers/api/v1/secure_controller.rb
- app/models/user.rb
- config/routes.rb
- db/migrate/20260808154603_create_users.rb
- db/schema.rb
- test/models/user_test.rb


## 14. Run RuboCop

RuboCop provides deterministic static-analysis findings.

In [15]:
offenses = run_rubocop(
    project_path=repo_path,
    files=ruby_files,
)

print(f"RuboCop found {len(offenses)} offenses.")

RuboCop found 3 offenses.


## 15. Display RuboCop findings

In [16]:
rubocop_context = format_offenses(offenses)

print(rubocop_context)


File: app/controllers/api/v1/auth_controller.rb
Line: 13
Severity: convention
Cop: Layout/TrailingEmptyLines
Message: Final newline missing.


File: app/controllers/api/v1/secure_controller.rb
Line: 4
Severity: convention
Cop: Layout/TrailingEmptyLines
Message: Final newline missing.


File: config/routes.rb
Line: 10
Severity: convention
Cop: Style/StringLiterals
Message: Prefer double-quoted strings unless you need single quotes to avoid extra backslashes for escaping.



## 16. Convert RuboCop findings to the common review schema

In [17]:
static_issues = rubocop_to_issues(offenses)

print(f"Normalized static issues: {len(static_issues)}")

for issue in static_issues:
    print(
        f"[{issue.severity.upper()}] "
        f"{issue.title} - "
        f"{issue.file}:{issue.line}"
    )

Normalized static issues: 3
[LOW] RuboCop: Layout/TrailingEmptyLines - app/controllers/api/v1/auth_controller.rb:13
[LOW] RuboCop: Layout/TrailingEmptyLines - app/controllers/api/v1/secure_controller.rb:4
[LOW] RuboCop: Style/StringLiterals - config/routes.rb:10


## 17. Run the AI review

The LLM receives the changed Ruby code, retrieved RAG context, and RuboCop findings.

In [18]:
ai_result = review(
    changed_code=changed_code,
    db=db,
    rubocop_context=rubocop_context,
)

print("AI review completed.")

AI review completed.


## 18. Inspect the AI result

In [19]:
print("SUMMARY")
print(ai_result.summary)
print()
print("SCORE:", ai_result.score)
print("ISSUES:", len(ai_result.issues))
print()

for issue in ai_result.issues:
    print(
        f"[{issue.severity.upper()}] "
        f"{issue.title}"
    )
    print("Category:", issue.category)
    print("File:", issue.file)
    print("Line:", issue.line)
    print("Explanation:", issue.explanation)
    print("Recommendation:", issue.recommendation)
    print("Evidence:", ", ".join(issue.evidence))
    print("-" * 70)

SUMMARY
The PR introduces basic user authentication, but the login endpoint has a critical control-flow bug and the route is wired to the wrong controller action. There are also missing model-level safeguards and no tests for the new authentication behavior.

SCORE: 35
ISSUES: 4

[CRITICAL] Login action always returns unauthorized even after successful authentication
Category: security
File: app/controllers/api/v1/auth_controller.rb
Line: 6
Explanation: The controller renders a success response inside the `if`, but it does not `return` or use an `else`, so execution continues and the unauthorized response is rendered afterward. In practice, a successful login still ends up as an unauthorized response, breaking authentication entirely.
Recommendation: Make the unauthorized render conditional, e.g. `return render json: { user: user }, status: :ok if user&.authenticate(...)` followed by a single unauthorized render, or use an `else` branch.
Evidence: `if user && user.authenticate(params[:

## 19. Merge AI + RuboCop findings

The aggregator removes obvious duplicate findings.

In [20]:
final_result = merge_review(
    ai_result=ai_result,
    static_issues=static_issues,
)

print("Final review assembled.")

Final review assembled.


## 20. Final human-readable report

In [21]:
print("=" * 80)
print("RUBY AI PULL REQUEST REVIEW")
print("=" * 80)
            
print()
print("PR:", PR_URL)
print("Repository:", f"{pr_info['owner']}/{pr_info['repo']}")
print("PR number:", pr_info["number"])
print("Changed files:", len(parsed["changed_files"]))
print("Ruby files:", len(parsed["ruby_files"]))
print("Test files:", len(parsed["test_files"]))
print("RuboCop offenses:", len(offenses))
            
print()
print("SCORE:", f"{final_result.score}/100")
print()
print("SUMMARY")
print(final_result.summary)
print()
print("ISSUES")
print("=" * 80)

for index, issue in enumerate(final_result.issues, start=1):
    print(f"\n{index}. [{issue.severity.upper()}] {issue.title}")
    print(f"   Category: {issue.category}")
    print(f"   File: {issue.file}")
    print(f"   Line: {issue.line}")
    print(f"   Explanation: {issue.explanation}")
    print(f"   Recommendation: {issue.recommendation}")
    print(f"   Evidence: {', '.join(issue.evidence)}")

print()
print("POSITIVE FINDINGS")
for finding in final_result.positive_findings:
    print("-", finding)

RUBY AI PULL REQUEST REVIEW

PR: https://github.com/renzodiaz/notes-api/pull/4
Repository: renzodiaz/notes-api
PR number: 4
Changed files: 10
Ruby files: 7
Test files: 1
RuboCop offenses: 3

SCORE: 35/100

SUMMARY
The PR introduces basic user authentication, but the login endpoint has a critical control-flow bug and the route is wired to the wrong controller action. There are also missing model-level safeguards and no tests for the new authentication behavior.

ISSUES

1. [LOW] RuboCop: Layout/TrailingEmptyLines
   Category: style
   File: app/controllers/api/v1/auth_controller.rb
   Line: 13
   Explanation: Final newline missing.
   Recommendation: Apply the recommended RuboCop correction.
   Evidence: RuboCop

2. [LOW] RuboCop: Layout/TrailingEmptyLines
   Category: style
   File: app/controllers/api/v1/secure_controller.rb
   Line: 4
   Explanation: Final newline missing.
   Recommendation: Apply the recommended RuboCop correction.
   Evidence: RuboCop

3. [LOW] RuboCop: Style/Strin

## 21. Export the final result as JSON

This will later be useful for the evaluation benchmark and Gradio UI.

In [22]:
result_json = final_result.model_dump_json(indent=2)
print(result_json)

{
  "summary": "The PR introduces basic user authentication, but the login endpoint has a critical control-flow bug and the route is wired to the wrong controller action. There are also missing model-level safeguards and no tests for the new authentication behavior.",
  "score": 35,
  "issues": [
    {
      "title": "RuboCop: Layout/TrailingEmptyLines",
      "severity": "low",
      "category": "style",
      "file": "app/controllers/api/v1/auth_controller.rb",
      "line": 13,
      "explanation": "Final newline missing.",
      "recommendation": "Apply the recommended RuboCop correction.",
      "evidence": [
        "RuboCop"
      ]
    },
    {
      "title": "RuboCop: Layout/TrailingEmptyLines",
      "severity": "low",
      "category": "style",
      "file": "app/controllers/api/v1/secure_controller.rb",
      "line": 4,
      "explanation": "Final newline missing.",
      "recommendation": "Apply the recommended RuboCop correction.",
      "evidence": [
        "RuboCop"
  

## 22. Simple statistics for this PR

In [23]:
severity_counts = {}
category_counts = {}

for issue in final_result.issues:
    severity_counts[issue.severity] = severity_counts.get(issue.severity, 0) + 1
    category_counts[issue.category] = category_counts.get(issue.category, 0) + 1

print("Severity distribution:")
for severity, count in sorted(severity_counts.items()):
    print(f"- {severity}: {count}")

print()
print("Category distribution:")
for category, count in sorted(category_counts.items()):
    print(f"- {category}: {count}")

Severity distribution:
- critical: 1
- high: 2
- low: 3
- medium: 1

Category distribution:
- rails: 1
- security: 2
- style: 3
- testing: 1


## Pipeline complete

The current system is:

```text
GitHub PR
   │
   ├── .diff ──→ Diff Parser ──→ Changed Ruby Code
   │                                  │
   │                                  ▼
   │                              RAG / FAISS
   │                                  │
   │                                  ▼
   └── Repository ──→ RuboCop ─────→ OpenAI
                                      │
                                      ▼
                               Structured Review
                                      │
                                      ▼
                                  Final Report
```

Next milestone: create a controlled benchmark with intentionally flawed Ruby PRs and calculate precision, recall, F1, false-positive rate, and review time.